In [1]:
import os
os.chdir('/home/smallyan/eval_agent')
print(f"Working directory: {os.getcwd()}")

Working directory: /home/smallyan/eval_agent


In [2]:
# Explore the repository structure
repo_path = '/net/scratch2/smallyan/leela-logit-lens_eval'

import os

def list_dir_tree(path, prefix="", max_depth=3, current_depth=0):
    """List directory tree structure"""
    if current_depth >= max_depth:
        return
    
    try:
        items = sorted(os.listdir(path))
    except PermissionError:
        return
    
    dirs = []
    files = []
    for item in items:
        if item.startswith('.'):
            continue
        full_path = os.path.join(path, item)
        if os.path.isdir(full_path):
            dirs.append(item)
        else:
            files.append(item)
    
    for f in files:
        print(f"{prefix}{f}")
    
    for d in dirs:
        print(f"{prefix}{d}/")
        list_dir_tree(os.path.join(path, d), prefix + "  ", max_depth, current_depth + 1)

print("Repository Structure:")
print("=" * 50)
list_dir_tree(repo_path)

Repository Structure:
768x15x24h-t82-swa-7464000.pb
768x15x24h-t82-swa-7464000.pb.gz
CodeWalkthrough.md
documentation.pdf
plan.md
pyproject.toml
bash_scripts/
  evaluate_concepts.sh
  evaluate_puzzles.sh
  install_bayeselo.sh
  install_stockfish.sh
  run_tournament.sh
data/
evaluation/
  consistency_evaluation.json
  self_matching.ipynb
  models/
    768x15x24h-t77-swa-5230000.pb.gz
iteration_model/
  LD2.onnx
  interesting_puzzles.pkl
  lc0-original.onnx
  lc0-random.onnx
  lc0.onnx
  unfiltered_puzzles.pkl
lc0_bin/
  lc0.tar.gz
lczero-common/
  proto/
    chunk.proto
    net.proto
lczero_proto/
  net_pb2.py
  __pycache__/
    net_pb2.cpython-311.pyc
notebooks/
  demo.ipynb
  figure1.ipynb
  forgotten_puzzle_figure.ipynb
  policy_metrics.ipynb
  puzzle_results.ipynb
  tournament_results.ipynb
scripts/
  evaluate_concepts.py
  evaluate_puzzles.py
  puzzle_history_augmentation.py
  tournament.py
src/
  leela_logit_lens/
    __init__.py
    __pycache__/
    core/
    tools/
    tournamen

In [3]:
# Check available data and results
import os

repo_path = '/net/scratch2/smallyan/leela-logit-lens_eval'

# Check data directory
print("Data directory:")
data_path = os.path.join(repo_path, 'data')
if os.path.exists(data_path):
    for item in os.listdir(data_path):
        full_path = os.path.join(data_path, item)
        if os.path.isfile(full_path):
            size = os.path.getsize(full_path) / (1024 * 1024)
            print(f"  {item} ({size:.2f} MB)")
        else:
            print(f"  {item}/ (directory)")
else:
    print("  Data directory not found")

# Check results directory
print("\nResults directory:")
results_path = os.path.join(repo_path, 'results')
if os.path.exists(results_path):
    for item in os.listdir(results_path):
        full_path = os.path.join(results_path, item)
        size = os.path.getsize(full_path) / (1024 * 1024) if os.path.isfile(full_path) else 0
        print(f"  {item} ({size:.2f} MB)" if os.path.isfile(full_path) else f"  {item}/")
else:
    print("  Results directory not found")

# Check iteration_model directory  
print("\nIteration model directory:")
iter_path = os.path.join(repo_path, 'iteration_model')
if os.path.exists(iter_path):
    for item in os.listdir(iter_path):
        full_path = os.path.join(iter_path, item)
        size = os.path.getsize(full_path) / (1024 * 1024) if os.path.isfile(full_path) else 0
        print(f"  {item} ({size:.2f} MB)" if os.path.isfile(full_path) else f"  {item}/")

Data directory:

Results directory:
  Results directory not found

Iteration model directory:
  interesting_puzzles.pkl (10.67 MB)
  lc0.onnx (361.13 MB)
  lc0-random.onnx (361.13 MB)
  LD2.onnx (14.19 MB)
  unfiltered_puzzles.pkl (364.51 MB)
  lc0-original.onnx (361.13 MB)


In [4]:
# Check for puzzles.csv in data directory
import os

repo_path = '/net/scratch2/smallyan/leela-logit-lens_eval'

# Check if data directory has files
data_path = os.path.join(repo_path, 'data')
print("Checking data directory contents:")
if os.path.exists(data_path):
    items = os.listdir(data_path)
    print(f"Items found: {items}")
    for item in items:
        full_path = os.path.join(data_path, item)
        if os.path.isfile(full_path):
            size = os.path.getsize(full_path) / (1024 * 1024)
            print(f"  {item} ({size:.2f} MB)")
        else:
            subitems = os.listdir(full_path)[:5]  # First 5 items
            print(f"  {item}/ (directory) - {len(os.listdir(full_path))} items")
else:
    print("  Data directory doesn't exist or is empty")

# Check for puzzle data in iteration_model
iter_path = os.path.join(repo_path, 'iteration_model')
print(f"\nChecking iteration_model for puzzle files:")
if os.path.exists(iter_path):
    for f in os.listdir(iter_path):
        if 'puzzle' in f.lower():
            size = os.path.getsize(os.path.join(iter_path, f)) / (1024 * 1024)
            print(f"  {f} ({size:.2f} MB)")

Checking data directory contents:
Items found: []

Checking iteration_model for puzzle files:
  interesting_puzzles.pkl (10.67 MB)
  unfiltered_puzzles.pkl (364.51 MB)


In [5]:
# Let's check GPU availability and load the puzzle data
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nUsing device: {device}")

CUDA available: True
GPU: NVIDIA A100 80GB PCIe
GPU Memory: 85.09 GB

Using device: cuda


# Replication of Leela Logit Lens Experiment

## Goal
Replicate the experiment analyzing how Leela Chess Zero (LC0) progressively builds understanding across layers using the "logit lens" technique extended to Post-LN transformer architectures.

## Key Findings from Original Plan
1. Neural networks perform iterative inference with capability progression in distinct phases
2. Three-phase progression: early rapid gains (through layer 5), middle plateau (through layer 10), late strengthening (layer 11+)
3. Layer-wise concept preferences evolve from aggressive to balanced play

In [6]:
# Load puzzle data from iteration_model directory
import pickle
import pandas as pd
import sys

sys.path.insert(0, '/net/scratch2/smallyan/leela-logit-lens_eval/src')

puzzle_path = '/net/scratch2/smallyan/leela-logit-lens_eval/iteration_model/interesting_puzzles.pkl'

with open(puzzle_path, 'rb') as f:
    puzzles = pickle.load(f)

print(f"Loaded puzzles: {type(puzzles)}")
if isinstance(puzzles, pd.DataFrame):
    print(f"Shape: {puzzles.shape}")
    print(f"Columns: {puzzles.columns.tolist()}")
    print(f"\nFirst few entries:")
    print(puzzles.head())

Loaded puzzles: <class 'pandas.core.frame.DataFrame'>
Shape: (22517, 19)
Columns: ['PuzzleId', 'FEN', 'Moves', 'Rating', 'RatingDeviation', 'Popularity', 'NbPlays', 'Themes', 'GameUrl', 'OpeningTags', 'principal_variation', 'full_pv_probs', 'full_model_moves', 'full_wdl', 'sparring_full_pv_probs', 'sparring_full_model_moves', 'sparring_wdl', 'different_targets', 'corrupted_fen']

First few entries:
    PuzzleId                                                FEN  \
22     001w5  1rb2rk1/q5P1/4p2p/3p3p/3P1P2/2P5/2QK3P/3R2R1 b...   
111    006wz  2r5/4ppkp/5bp1/1p6/1P6/P3B3/2r2PPP/1R1R2K1 b -...   
116    00761  3r2k1/1b3pbR/p2P2P1/3p2N1/2p5/2P2N2/PP6/2K5 b ...   
170    00AoZ       8/1R6/p1pk4/6bp/1QP5/P7/KP6/3r2q1 b - - 2 44   
182    00Bg4  3r2k1/1q3ppp/p2rp3/Qp1B4/7P/P4P2/1PP3P1/1K1R3R...   

                             Moves  Rating  RatingDeviation  Popularity  \
22            f8f7 c2h7 g8h7 g7g8q    1073               77          91   
111  f6b2 b1b2 c2b2 e3d4 f7f6 d4b2    1515   

## Implementation

### Part 1: Model Loading and Logit Lens Setup

We'll implement the core logit lens functionality from first principles based on understanding the original codebase.

In [7]:
# Load the LC0 model using the leela_interp library
from leela_interp import Lc0sight, LeelaBoard

# Use the original model (not finetuned) with position history
model_path = '/net/scratch2/smallyan/leela-logit-lens_eval/iteration_model/lc0-original.onnx'

# Load model on GPU
model = Lc0sight(path=model_path, device='cuda')
model.eval()

print(f"Model loaded successfully!")
print(f"Number of layers: {model.N_LAYERS}")
print(f"Embedding dimension: {model.D_MODEL}")
print(f"Device: {model.device}")

Using device: cuda


Model loaded successfully!
Number of layers: 15
Embedding dimension: 768
Device: cuda


### Part 2: Reimplementing the Logit Lens

The logit lens technique for Post-LN transformers works by:
1. Running the model up to layer ℓ
2. Applying zero ablation to all sublayer outputs from layer ℓ onwards
3. Preserving layer normalizations but zeroing their biases
4. Reading the policy head output from this ablated state

This allows us to see what the model "knows" at each intermediate layer.